# GA1-220501094-01-AA1-EV07 – Preparación, limpieza y estructuración de datos
## Zona Fit Evolution: análisis de deserción de miembros

**Aprendiz:** Juan Diego Mejía Llano
**Programa:** Análisis y Desarrollo de Software (ADSO) – SENA Regional Antioquia, ficha 3223873

**Pregunta analítica del proyecto:** ¿Qué factores (tipo de plan, frecuencia de asistencia, horario de entrenamiento y antigüedad) influyen en que un miembro de Zona Fit Evolution cancele o no renueve su membresía durante sus primeros seis meses?

**Pregunta orientadora de la actividad:** ¿Cómo preparar y estructurar los datos del proyecto individual para obtener información confiable que permita responder la pregunta analítica planteada?

Este notebook aplica las técnicas de las lecciones 2, 3 y 4 del material *Preparar y estructurar datos con Python* a los datasets del proyecto. Parte del diagnóstico de calidad realizado en `notebooks/data_quality.ipynb` y documentado en `reports/data_quality.md`.

| Sección | Contenido |
|---|---|
| 0. Preparación | Carga de los datasets originales y estado inicial de la calidad |
| Lección 2 | Limpieza y estandarización: duplicados, tipos, faltantes, texto, reglas de negocio, pipeline y validación |
| Lección 3 | Filtrado, ordenamiento y agrupación: indicadores para la pregunta analítica |
| Lección 4 | Combinación y estructuración: relaciones, tablas consolidadas, integridad y exportación |
| Cierre | Comparación de la calidad antes y después y conclusiones |

# 0. Preparación del entorno y carga de los datos

Los datasets originales están en `data/raw/` y **no se modifican**: todas las transformaciones se hacen sobre copias en memoria y los resultados se guardan en `data/processed/`.

`FECHA_CORTE` es la fecha de la exportación de la base de datos de Zona Fit (15 de octubre de 2026). Se usa para saber qué membresías ya vencieron y cuáles todavía se pueden observar.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

RUTA_RAW = Path("../data/raw")
RUTA_PROCESSED = Path("../data/processed")
FECHA_CORTE = pd.Timestamp("2026-10-15")

print("Versión de pandas:", pd.__version__)

Versión de pandas: 3.0.6


In [2]:
# Zona Fit Evolution (fuente primaria, simulada)
zf_planes = pd.read_csv(RUTA_RAW / "zonafit" / "planes.csv")
zf_miembros = pd.read_csv(RUTA_RAW / "zonafit" / "miembros.csv")
zf_membresias = pd.read_csv(RUTA_RAW / "zonafit" / "membresias.csv")
zf_pagos = pd.read_csv(RUTA_RAW / "zonafit" / "pagos.csv")
zf_accesos = pd.read_csv(RUTA_RAW / "zonafit" / "accesos.csv")

# Kaggle (fuente secundaria)
kg_churn = pd.read_csv(RUTA_RAW / "kaggle" / "gym_churn_us.csv")
kg_actividad = pd.read_csv(RUTA_RAW / "kaggle" / "gym_attendance_cleaned_data.csv")

# datos.gov.co (fuente gubernamental)
gov_escenarios = pd.read_csv(RUTA_RAW / "datos_gov" / "escenarios_deportivos_inder.csv")

datos_originales = {
    "zf_planes": zf_planes,
    "zf_miembros": zf_miembros,
    "zf_membresias": zf_membresias,
    "zf_pagos": zf_pagos,
    "zf_accesos": zf_accesos,
    "kg_churn": kg_churn,
    "kg_actividad": kg_actividad,
    "gov_escenarios": gov_escenarios,
}

for nombre, df in datos_originales.items():
    print(f"{nombre:<16} {df.shape[0]:>6} filas x {df.shape[1]:>2} columnas")

zf_planes             6 filas x  6 columnas
zf_miembros         908 filas x  7 columnas
zf_membresias      3425 filas x  7 columnas
zf_pagos           3572 filas x  6 columnas
zf_accesos        57003 filas x  5 columnas
kg_churn           4000 filas x 14 columnas
kg_actividad       2600 filas x 10 columnas
gov_escenarios     1071 filas x  5 columnas


## Estado inicial de la calidad

La función `resumen_calidad` resume cada dataset en una fila: filas, nulos, filas duplicadas, columnas de texto y memoria. Se ejecuta ahora sobre los datos originales y al final de la limpieza sobre los datos limpios, para comparar el antes y el después.

In [3]:
def resumen_calidad(datasets):
    """Devuelve una fila por dataset con indicadores básicos de calidad."""
    filas = []
    for nombre, df in datasets.items():
        filas.append({
            "dataset": nombre,
            "filas": len(df),
            "columnas": df.shape[1],
            "nulos": int(df.isna().sum().sum()),
            "filas_duplicadas": int(df.duplicated().sum()),
            "columnas_texto": int(sum(df[c].dtype == "str" for c in df.columns)),
            "memoria_kb": round(df.memory_usage(deep=True).sum() / 1024, 1),
        })
    return pd.DataFrame(filas)


estado_inicial = resumen_calidad(datos_originales)
estado_inicial

,dataset,filas,columnas,nulos,filas_duplicadas,columnas_texto,memoria_kb
0,zf_planes,6,6,5,0,2,0.6
1,zf_miembros,908,7,127,0,6,95.1
2,zf_membresias,3425,7,51,0,3,277.4
3,zf_pagos,3572,6,108,12,4,276.1
4,zf_accesos,57003,5,1140,0,3,4283.0
5,kg_churn,4000,14,0,0,0,437.6
6,kg_actividad,2600,10,0,0,6,317.0
7,gov_escenarios,1071,5,0,0,3,128.7


# Lección 2. Limpieza y estandarización de datos

Se resuelven los problemas encontrados en el diagnóstico siguiendo el orden del material: **duplicados → tipos de dato → valores faltantes → texto y categorías → reglas de negocio → datasets externos**. Para cada problema se muestra la evidencia, se define una **función reutilizable** que lo corrige y se justifica la decisión (eliminar, corregir o conservar).

Las funciones reciben un DataFrame y devuelven otro, sin modificar el original. Al final se encadenan en un **pipeline con `pipe()`** que registra los cambios de cada paso y se valida con `assert`.

## 2.1 Registros duplicados

### a) Miembros inscritos dos veces (`zf_miembros`)

In [4]:
repetidos = zf_miembros[zf_miembros["documento_hash"].duplicated(keep=False)]
print("Registros con documento repetido:", len(repetidos))
print("Personas afectadas:", repetidos["documento_hash"].nunique())
print("IDs de las copias:", sorted(zf_miembros.loc[zf_miembros["documento_hash"].duplicated(), "id_miembro"]))

sin_membresia = ~zf_miembros["id_miembro"].isin(zf_membresias["id_miembro"])
print("¿Las copias tienen membresías o accesos?:",
      zf_membresias["id_miembro"].isin(range(901, 909)).any(),
      zf_accesos["id_miembro"].isin(range(901, 909)).any())

Registros con documento repetido: 16
Personas afectadas: 8
IDs de las copias: [901, 902, 903, 904, 905, 906, 907, 908]
¿Las copias tienen membresías o accesos?: False False


In [5]:
def quitar_miembros_duplicados(df):
    """Conserva el primer registro (el id más bajo) de cada documento."""
    return (df.sort_values("id_miembro")
              .drop_duplicates(subset="documento_hash", keep="first")
              .reset_index(drop=True))


print("Antes:", zf_miembros.shape, "| Después:", quitar_miembros_duplicados(zf_miembros).shape)

Antes: (908, 7) | Después: (900, 7)


**Decisión: eliminar.** Las 8 personas repetidas tienen los mismos datos en los dos registros, y las copias (IDs 901 a 908) no tienen membresías ni accesos. Se conserva el registro original (el de id más bajo), así que no se pierde ninguna relación con las otras tablas. El duplicado se detecta por `documento_hash` y no por la fila completa, porque el `id_miembro` es distinto en cada copia.

### b) Pagos duplicados (`zf_pagos`)

In [6]:
print("Filas idénticas:", zf_pagos.duplicated().sum())
print("id_pago repetidos:", zf_pagos["id_pago"].duplicated().sum())
display(zf_pagos[zf_pagos.duplicated(keep=False)].sort_values("id_pago").head(4))


def quitar_filas_duplicadas(df):
    """Elimina filas exactamente iguales y renumera el índice."""
    return df.drop_duplicates().reset_index(drop=True)


print("Antes:", zf_pagos.shape, "| Después:", quitar_filas_duplicadas(zf_pagos).shape)

Filas idénticas: 12
id_pago repetidos: 12


,id_pago,id_membresia,fecha_pago,valor,metodo_pago,estado_pago
174,175,163,2026-09-27,90000,Efectivo,Aprobado
3571,175,163,2026-09-27,90000,Efectivo,Aprobado
3569,696,661,2026-08-11,240000,Efectivo,Aprobado
695,696,661,2026-08-11,240000,Efectivo,Aprobado


Antes: (3572, 6) | Después: (3560, 6)


**Decisión: eliminar.** Las 12 filas repetidas son idénticas, incluido el `id_pago`, que debería ser único. Si se conservan, los ingresos quedan inflados.

Las **lecturas dobles del lector de huella** también son duplicados, pero para detectarlas hay que comparar fechas y horas, así que se tratan en la sección 2.5, después de convertir los tipos.

## 2.2 Tipos de datos

En la carga, 6 columnas de fecha de Zona Fit y el monto de los pagos quedaron como texto (`str`). Sin convertirlas no se puede calcular la antigüedad, la frecuencia ni la franja horaria, ni sumar los pagos.

In [7]:
columnas_fecha = {
    "zf_miembros": ["fecha_nacimiento", "fecha_registro"],
    "zf_membresias": ["fecha_inicio", "fecha_fin"],
    "zf_pagos": ["fecha_pago"],
    "zf_accesos": ["fecha_hora"],
}
for nombre, columnas in columnas_fecha.items():
    for columna in columnas:
        print(f"{nombre}.{columna:<17} {datos_originales[nombre][columna].dtype}")

print("zf_pagos.valor               ", zf_pagos["valor"].dtype)
print("Ejemplos de valor no numérico:", zf_pagos.loc[zf_pagos["valor"].str.contains("$", regex=False), "valor"].unique()[:4])

formato_iso = zf_pagos["fecha_pago"].str.match(r"^\d{4}-\d{2}-\d{2}$")
print("Fechas de pago AAAA-MM-DD:", formato_iso.sum(), "| DD/MM/AAAA:", (~formato_iso).sum())

zf_miembros.fecha_nacimiento  str
zf_miembros.fecha_registro    str
zf_membresias.fecha_inicio      str
zf_membresias.fecha_fin         str
zf_pagos.fecha_pago        str
zf_accesos.fecha_hora        str
zf_pagos.valor                str
Ejemplos de valor no numérico: <ArrowStringArray>
['$70.000', '$60.000', '$450.000', '$90.000']
Length: 4, dtype: str
Fechas de pago AAAA-MM-DD: 3430 | DD/MM/AAAA: 142


In [8]:
def convertir_fechas(df, columnas, formato="%Y-%m-%d"):
    """Convierte columnas de texto a fecha con un formato explícito."""
    df = df.copy()
    for columna in columnas:
        df[columna] = pd.to_datetime(df[columna], format=formato)
    return df


def convertir_fecha_pago(df):
    """Convierte fecha_pago, que viene en dos formatos: AAAA-MM-DD y DD/MM/AAAA."""
    df = df.copy()
    iso = df["fecha_pago"].str.match(r"^\d{4}-\d{2}-\d{2}$")
    df["fecha_pago"] = pd.to_datetime(df["fecha_pago"].where(iso), format="%Y-%m-%d").fillna(
        pd.to_datetime(df["fecha_pago"].where(~iso), format="%d/%m/%Y"))
    return df


def limpiar_valor(df):
    """Quita el signo de pesos y el punto de miles y convierte valor a entero."""
    df = df.copy()
    df["valor"] = pd.to_numeric(df["valor"].str.replace(r"[$.]", "", regex=True)).astype("int64")
    return df


def ajustar_enteros(df, tipos):
    """Convierte columnas a tipos enteros más pequeños; 'Int16' admite nulos."""
    return df.astype(tipos)


prueba = zf_pagos.pipe(convertir_fecha_pago).pipe(limpiar_valor)
print(prueba.dtypes)
print("Fechas sin convertir:", prueba["fecha_pago"].isna().sum())
print("Valor total de pagos (sin quitar duplicados):", f"{prueba['valor'].sum():,}")

id_pago                  int64
id_membresia             int64
fecha_pago      datetime64[us]
valor                    int64
metodo_pago                str
estado_pago                str
dtype: object
Fechas sin convertir: 0
Valor total de pagos (sin quitar duplicados): 402,989,000


**Decisión: corregir.**

- Las fechas se convierten con `pd.to_datetime` y un **formato explícito**. En `fecha_pago` cada formato se convierte por separado: si se deja que pandas adivine, una fecha como `01/06/2026` puede quedar como 6 de enero en lugar de 1 de junio.
- `valor` se limpia quitando `$` y el punto de miles con una expresión regular (`[$.]`) y luego se convierte a número con `pd.to_numeric`.
- Los identificadores y conteos se pasan a enteros más pequeños (`int16`, `int32`), y `ingresos_incluidos` a `Int16` (con mayúscula), el entero que admite nulos, porque los planes ilimitados no tienen límite de ingresos.

## 2.3 Valores faltantes

Antes de decidir, se revisa cuántos faltan y qué significa cada vacío.

In [9]:
faltantes = []
for nombre, df in datos_originales.items():
    conteo = df.isna().sum()
    for columna, cantidad in conteo[conteo > 0].items():
        faltantes.append({"dataset": nombre, "columna": columna, "faltantes": int(cantidad),
                          "porcentaje": round(cantidad / len(df) * 100, 2)})
pd.DataFrame(faltantes)

,dataset,columna,faltantes,porcentaje
0,zf_planes,ingresos_incluidos,5,83.33
1,zf_miembros,genero,27,2.97
2,zf_miembros,fecha_nacimiento,54,5.95
3,zf_miembros,comuna,46,5.07
4,zf_membresias,precio_pagado,51,1.49
5,zf_pagos,metodo_pago,108,3.02
6,zf_accesos,dispositivo,1140,2.00


In [10]:
print("Fechas de nacimiento de relleno (1900-01-01):", (zf_miembros["fecha_nacimiento"] == "1900-01-01").sum())

precio_nulo = zf_membresias[zf_membresias["precio_pagado"].isna()]
pagos_validos = (zf_pagos.pipe(quitar_filas_duplicadas).pipe(limpiar_valor)
                 .query("estado_pago == 'Aprobado' and valor > 0"))
print("Membresías sin precio_pagado:", len(precio_nulo))
print("  ... que tienen pagos aprobados:", precio_nulo["id_membresia"].isin(pagos_validos["id_membresia"]).sum())

Fechas de nacimiento de relleno (1900-01-01): 9
Membresías sin precio_pagado: 51
  ... que tienen pagos aprobados: 49


In [11]:
def rellenar_sin_dato(df, columnas):
    """Reemplaza los vacíos de columnas categóricas por la categoría 'Sin dato'."""
    df = df.copy()
    df[columnas] = df[columnas].fillna("Sin dato")
    return df


def anular_fecha_relleno(df):
    """La fecha 1900-01-01 es un valor de relleno: se convierte en faltante (NaT)."""
    df = df.copy()
    df["fecha_nacimiento"] = df["fecha_nacimiento"].mask(df["fecha_nacimiento"] == pd.Timestamp("1900-01-01"))
    return df


def completar_precio(df, pagos, planes):
    """Completa precio_pagado con los pagos aprobados de la membresía; si no hay, con el precio de lista."""
    df = df.copy()
    pagado = (pagos.query("estado_pago == 'Aprobado' and valor > 0")
                   .groupby("id_membresia")["valor"].sum())
    precio_lista = df["id_plan"].map(planes.set_index("id_plan")["precio"])
    df["precio_pagado"] = (df["precio_pagado"]
                           .fillna(df["id_membresia"].map(pagado))
                           .fillna(precio_lista)
                           .astype("int64"))
    return df


def completar_dispositivo(df):
    """La sede tiene un solo lector: los vacíos de dispositivo corresponden a LECTOR-01."""
    df = df.copy()
    df["dispositivo"] = df["dispositivo"].fillna("LECTOR-01")
    return df

| Columna | Faltantes | Decisión | Justificación |
|---|---:|---|---|
| `zf_planes.ingresos_incluidos` | 5 | **Conservar** | No es un error: los planes ilimitados no tienen límite de ingresos |
| `zf_miembros.genero` | 27 | **Rellenar con "Sin dato"** | No se puede inferir; borrar al miembro perdería sus membresías |
| `zf_miembros.comuna` | 46 | **Rellenar con "Sin dato"** | Mismo criterio: se conserva el miembro y el vacío queda visible como categoría |
| `zf_miembros.fecha_nacimiento` | 54 + 9 de relleno | **Conservar como faltante** | Inventar una edad sesgaría el análisis; la fecha `1900-01-01` (126 años) también pasa a faltante |
| `zf_membresias.precio_pagado` | 51 | **Completar** | 49 tienen pagos aprobados y se completan con la suma de esos pagos; las otras 2 con el precio de lista del plan |
| `zf_pagos.metodo_pago` | 108 | **Rellenar con "Sin dato"** | El pago existe y tiene valor; solo falta el medio |
| `zf_accesos.dispositivo` | 1.140 | **Completar con `LECTOR-01`** | La sede tiene un único lector, así que el dato se puede deducir |

Ningún faltante supera el 6 % en columnas relevantes y ninguno se elimina: en todos los casos el registro tiene información útil para la pregunta.

## 2.4 Texto y categorías

`genero` tiene 6 formas de escribir 2 valores y `comuna` tiene `laureles` en lugar de `Laureles-Estadio`. Si no se unifican, el mismo grupo queda dividido al agrupar.

In [12]:
print(zf_miembros["genero"].value_counts(dropna=False))
print()
print("Comunas escritas como 'laureles':", (zf_miembros["comuna"] == "laureles").sum())

texto_zonafit = {nombre: df.select_dtypes(include="str").columns.tolist()
                 for nombre, df in datos_originales.items() if nombre.startswith("zf_")}
for nombre, columnas in texto_zonafit.items():
    df = datos_originales[nombre]
    con_espacios = sum(((df[c].str.strip() != df[c]) & df[c].notna()).sum() for c in columnas)
    print(f"{nombre:<14} valores con espacios al inicio o al final: {con_espacios}")

genero
M            446
F            402
NaN           27
f             15
Masculino      7
masculino      6
Femenino       5
Name: count, dtype: int64

Comunas escritas como 'laureles': 18
zf_planes      valores con espacios al inicio o al final: 0
zf_miembros    valores con espacios al inicio o al final: 0
zf_membresias  valores con espacios al inicio o al final: 0
zf_pagos       valores con espacios al inicio o al final: 0
zf_accesos     valores con espacios al inicio o al final: 0


In [13]:
MAPA_GENERO = {"M": "M", "Masculino": "M", "masculino": "M",
               "F": "F", "f": "F", "Femenino": "F"}
MAPA_COMUNA = {"laureles": "Laureles-Estadio"}


def estandarizar_texto(df, columnas):
    """Quita espacios al inicio y al final y une los espacios repetidos."""
    df = df.copy()
    for columna in columnas:
        df[columna] = df[columna].str.strip().str.replace(r"\s+", " ", regex=True)
    return df


def estandarizar_miembros(df):
    """Unifica las categorías de genero y comuna."""
    df = df.copy()
    df["genero"] = df["genero"].map(MAPA_GENERO)
    df["comuna"] = df["comuna"].replace(MAPA_COMUNA)
    return df


def a_categoria(df, columnas):
    """Convierte columnas de texto con pocos valores distintos al tipo category."""
    df = df.copy()
    df[columnas] = df[columnas].astype("category")
    return df


prueba = estandarizar_miembros(zf_miembros)
print(prueba["genero"].value_counts(dropna=False))
print("Comunas distintas antes:", zf_miembros["comuna"].nunique(), "| después:", prueba["comuna"].nunique())

genero
M      459
F      422
NaN     27
Name: count, dtype: int64
Comunas distintas antes: 13 | después: 12


Las tablas de Zona Fit no tienen espacios sobrantes al inicio o al final, así que la función `estandarizar_texto` solo se aplica en `gov_escenarios` (sección 2.6).

**Decisión: corregir.** `genero` se unifica con un diccionario y `map` (todo valor no previsto quedaría como faltante, lo que se comprueba en la validación), y `comuna` con `replace`. Las columnas con pocos valores distintos (género, comuna, canal, estados, método de pago, resultado) se convierten a `category`, que ocupa menos memoria y deja fijas las categorías válidas.

## 2.5 Reglas de negocio y consistencia

### a) Estado de las membresías

Regla del diccionario de datos: cuando pasa la fecha de fin, la membresía debe quedar `Vencida`.

In [14]:
membresias_fechas = convertir_fechas(zf_membresias, ["fecha_inicio", "fecha_fin"])
activa_vencida = (membresias_fechas["estado"] == "Activa") & (membresias_fechas["fecha_fin"] < FECHA_CORTE)
print("Membresías 'Activa' con fecha_fin anterior al corte:", activa_vencida.sum())
print(membresias_fechas["estado"].value_counts())


def recalcular_estado(df, fecha_corte=FECHA_CORTE):
    """Pasa a 'Vencida' toda membresía 'Activa' cuya fecha de fin ya pasó."""
    df = df.copy()
    vencida = (df["estado"] == "Activa") & (df["fecha_fin"] < fecha_corte)
    df["estado"] = df["estado"].mask(vencida, "Vencida")
    return df


print()
print(recalcular_estado(membresias_fechas)["estado"].value_counts())

Membresías 'Activa' con fecha_fin anterior al corte: 93
estado
Vencida      3027
Activa        394
Cancelada       4
Name: count, dtype: int64

estado
Vencida      3120
Activa        301
Cancelada       4
Name: count, dtype: int64


**Decisión: corregir.** Es el problema más crítico del diagnóstico, porque la deserción se mide con el estado de las membresías. Las 4 membresías `Cancelada` se conservan con su estado: son cancelaciones reales y cuentan como deserción.

### b) Accesos inválidos del lector de huella

Se aplican cuatro reglas, en este orden, y cada una se cuenta después de aplicar la anterior:

1. **Lecturas dobles:** dos lecturas del mismo miembro a menos de 60 segundos son el mismo ingreso.
2. **Miembros inexistentes:** accesos de IDs que no están en `zf_miembros`.
3. **Fuera de horario:** la sede abre de 5:00 a 22:00.
4. **Membresía cancelada:** el lector solo debe abrir con una membresía activa.

In [15]:
def quitar_lecturas_dobles(df, segundos=60):
    """Elimina las lecturas a menos de `segundos` del acceso anterior del mismo miembro."""
    ordenado = df.sort_values(["id_miembro", "fecha_hora"])
    diferencia = ordenado.groupby("id_miembro")["fecha_hora"].diff().dt.total_seconds()
    return ordenado[~(diferencia < segundos)].sort_values("id_acceso").reset_index(drop=True)


def quitar_miembros_inexistentes(df, miembros):
    """Conserva solo los accesos de miembros registrados."""
    return df[df["id_miembro"].isin(miembros["id_miembro"])].reset_index(drop=True)


def quitar_fuera_de_horario(df, apertura=5, cierre=22):
    """Conserva los accesos entre la hora de apertura y la de cierre."""
    hora = df["fecha_hora"].dt.hour
    return df[(hora >= apertura) & (hora < cierre)].reset_index(drop=True)


def asignar_membresia(df, membresias):
    """Agrega a cada acceso la membresía vigente ese día (no hay membresías superpuestas)."""
    cruce = (df.assign(dia=df["fecha_hora"].dt.normalize())
               .merge(membresias[["id_miembro", "id_membresia", "fecha_inicio", "fecha_fin"]],
                      on="id_miembro", how="left"))
    vigente = cruce[(cruce["dia"] >= cruce["fecha_inicio"]) & (cruce["dia"] <= cruce["fecha_fin"])]
    assert vigente["id_acceso"].is_unique, "Un acceso quedó con dos membresías vigentes"
    df = df.copy()
    df["id_membresia"] = df["id_acceso"].map(vigente.set_index("id_acceso")["id_membresia"]).astype("Int32")
    return df


def quitar_acceso_con_cancelada(df, membresias):
    """Elimina los accesos 'Permitido' durante una membresía cancelada."""
    estado = df["id_membresia"].map(membresias.set_index("id_membresia")["estado"])
    invalido = (df["resultado"] == "Permitido") & (estado == "Cancelada")
    return df[~invalido].reset_index(drop=True)


accesos_fechas = convertir_fechas(zf_accesos, ["fecha_hora"], formato="%Y-%m-%d %H:%M:%S")
paso_1 = quitar_lecturas_dobles(accesos_fechas)
paso_2 = quitar_miembros_inexistentes(paso_1, quitar_miembros_duplicados(zf_miembros))
paso_3 = quitar_fuera_de_horario(paso_2)
paso_4 = quitar_acceso_con_cancelada(asignar_membresia(paso_3, membresias_fechas), membresias_fechas)

print("Accesos originales:            ", len(accesos_fechas))
print("Lecturas dobles eliminadas:    ", len(accesos_fechas) - len(paso_1))
print("Miembros inexistentes:         ", len(paso_1) - len(paso_2))
print("Fuera de horario:              ", len(paso_2) - len(paso_3))
print("Permitidos con cancelada:      ", len(paso_3) - len(paso_4))
print("Accesos válidos:               ", len(paso_4))

Accesos originales:             57003
Lecturas dobles eliminadas:     675
Miembros inexistentes:          25
Fuera de horario:               15
Permitidos con cancelada:       179
Accesos válidos:                56109


In [16]:
sin_membresia_vigente = paso_4[paso_4["id_membresia"].isna()]
print("Accesos sin membresía vigente:", len(sin_membresia_vigente))
print(sin_membresia_vigente["resultado"].value_counts())

Accesos sin membresía vigente: 386
resultado
Denegado    386
Name: count, dtype: int64


**Decisión: eliminar** los cuatro tipos de accesos inválidos, porque inflan o distorsionan la frecuencia y el horario de asistencia, dos de los factores de la pregunta analítica:

- Las **675 lecturas dobles** son el problema de mayor volumen: una sola entrada contada dos veces.
- Los **25 accesos de miembros inexistentes** no se pueden atribuir a nadie.
- Los **15 accesos fuera de horario** (17 en el diagnóstico; 2 eran además lecturas dobles) ocurren con la sede cerrada.
- Los **179 accesos permitidos durante una membresía cancelada** (184 en el diagnóstico; 5 eran lecturas dobles) son asistencia de quien ya no es miembro.

Los accesos `Denegado` **se conservan**: todos ocurren sin membresía vigente, que es justo lo que dice la regla del lector, y muestran a personas que intentaron entrar después de vencer su plan. La función `asignar_membresia` deja en cada acceso la membresía vigente ese día, lo que se usará en el análisis.

**Pagos aprobados en cero:** se conservan, pero no se usan para completar precios. Son 14 pagos que el administrador debe revisar con caja; no afectan la deserción.

## 2.6 Datasets externos

Los datasets externos no se combinan con las tablas de Zona Fit (no comparten identificadores ni periodo), pero se limpian para usarlos como referencia.

- **`kg_churn`:** nombres de columna en `snake_case` y en minúscula, `Month_to_end_contract` a entero y las 7 variables binarias a `int8`.
- **`kg_actividad`:** `visit_date` a fecha, `check_in_time` a hora entera y se elimina `attendance_status`, que el diagnóstico mostró que no es confiable (1.350 ausencias con entrenamiento registrado).
- **`gov_escenarios`:** espacios repetidos en direcciones y nombres, y coordenadas fuera de Medellín convertidas en faltantes.

In [17]:
BINARIAS_CHURN = ["gender", "near_location", "partner", "promo_friends", "phone", "group_visits", "churn"]
MEDELLIN = {"latitud": (6.1, 6.4), "longitud": (-75.75, -75.45)}


def limpiar_kg_churn(df):
    """Normaliza nombres y tipos del dataset de deserción de Kaggle."""
    df = df.rename(columns=str.lower)
    return df.astype({**{c: "int8" for c in BINARIAS_CHURN},
                      "month_to_end_contract": "int8", "contract_period": "int8",
                      "age": "int8", "lifetime": "int16"})


def limpiar_kg_actividad(df):
    """Convierte fecha y hora y descarta attendance_status, que no es confiable."""
    df = convertir_fechas(df, ["visit_date"])
    df["check_in_hour"] = pd.to_datetime(df["check_in_time"], format="%H:%M:%S").dt.hour.astype("int8")
    return df.drop(columns=["attendance_status", "check_in_time"])


def limpiar_gov_escenarios(df):
    """Normaliza espacios y anula las coordenadas que caen fuera de Medellín."""
    df = estandarizar_texto(df, ["nombre_escenario", "direccion", "barrio"])
    dentro = (df["latitud"].between(*MEDELLIN["latitud"]) &
              df["longitud"].between(*MEDELLIN["longitud"]))
    df[["latitud", "longitud"]] = df[["latitud", "longitud"]].where(dentro)
    return df


print("kg_churn:", limpiar_kg_churn(kg_churn).dtypes.value_counts().to_dict())
print("kg_actividad columnas:", limpiar_kg_actividad(kg_actividad).columns.tolist())
escenarios_prueba = limpiar_gov_escenarios(gov_escenarios)
print("gov_escenarios: coordenadas anuladas:", escenarios_prueba["latitud"].isna().sum(),
      "| direcciones con espacios dobles:", escenarios_prueba["direccion"].str.contains("  ").sum())

kg_churn: {dtype('int8'): 10, dtype('float64'): 3, dtype('int16'): 1}


kg_actividad columnas: ['member_id', 'visit_date', 'age', 'gender', 'membership_type', 'workout_type', 'workout_duration_minutes', 'calories_burned', 'check_in_hour']


gov_escenarios: coordenadas anuladas: 22 | direcciones con espacios dobles: 0


En `kg_churn` los códigos 0/1 se conservan como números porque así los usan los modelos de deserción. Su significado documentado es: `near_location`, `partner`, `promo_friends`, `phone`, `group_visits` y `churn` valen 1 cuando la respuesta es *sí*. El dataset no indica qué valor de `gender` es hombre o mujer, así que esa columna no se usa para comparar por género.

Las coordenadas fuera de Medellín se anulan en lugar de borrar el escenario, porque la dirección y el barrio siguen siendo válidos.

## 2.7 Pipeline de limpieza con `pipe()` e informe de cambios

Las funciones anteriores se encadenan con `pipe()` partiendo de los datos originales. La función `anotar` toma una "foto" de la tabla después de cada paso (filas, nulos, duplicados y memoria) y la guarda en `registro`; como devuelve la misma tabla, se puede intercalar entre los pasos sin alterar el resultado.

El orden importa: `zf_pagos` se limpia antes que `zf_membresias` (para completar los precios) y `zf_accesos` al final (necesita los miembros y las membresías limpios).

In [18]:
registro = []


def anotar(df, tabla, paso):
    """Registra el estado de la tabla después de un paso y la devuelve sin cambios."""
    registro.append({
        "tabla": tabla,
        "paso": paso,
        "filas": len(df),
        "nulos": int(df.isna().sum().sum()),
        "duplicados": int(df.duplicated().sum()),
        "memoria_kb": round(df.memory_usage(deep=True).sum() / 1024, 1),
    })
    return df

In [19]:
planes_limpio = (zf_planes
    .pipe(anotar, "zf_planes", "0 original")
    .pipe(ajustar_enteros, {"id_plan": "int8", "duracion_dias": "int16",
                            "ingresos_incluidos": "Int16", "precio": "int32"})
    .pipe(a_categoria, ["tipo_plan"])
    .pipe(anotar, "zf_planes", "1 tipos"))

miembros_limpio = (zf_miembros
    .pipe(anotar, "zf_miembros", "0 original")
    .pipe(quitar_miembros_duplicados)
    .pipe(anotar, "zf_miembros", "1 sin duplicados")
    .pipe(convertir_fechas, ["fecha_nacimiento", "fecha_registro"])
    .pipe(anular_fecha_relleno)
    .pipe(ajustar_enteros, {"id_miembro": "int16"})
    .pipe(anotar, "zf_miembros", "2 tipos y fechas de relleno")
    .pipe(estandarizar_miembros)
    .pipe(rellenar_sin_dato, ["genero", "comuna"])
    .pipe(a_categoria, ["genero", "comuna", "canal_captacion"])
    .pipe(anotar, "zf_miembros", "3 categorías y faltantes"))

pagos_limpio = (zf_pagos
    .pipe(anotar, "zf_pagos", "0 original")
    .pipe(quitar_filas_duplicadas)
    .pipe(anotar, "zf_pagos", "1 sin duplicados")
    .pipe(limpiar_valor)
    .pipe(convertir_fecha_pago)
    .pipe(ajustar_enteros, {"id_pago": "int16", "id_membresia": "int16"})
    .pipe(anotar, "zf_pagos", "2 tipos")
    .pipe(rellenar_sin_dato, ["metodo_pago"])
    .pipe(a_categoria, ["metodo_pago", "estado_pago"])
    .pipe(anotar, "zf_pagos", "3 categorías y faltantes"))

membresias_limpio = (zf_membresias
    .pipe(anotar, "zf_membresias", "0 original")
    .pipe(convertir_fechas, ["fecha_inicio", "fecha_fin"])
    .pipe(ajustar_enteros, {"id_membresia": "int16", "id_miembro": "int16", "id_plan": "int8"})
    .pipe(anotar, "zf_membresias", "1 tipos")
    .pipe(completar_precio, pagos_limpio, planes_limpio)
    .pipe(anotar, "zf_membresias", "2 precio completado")
    .pipe(recalcular_estado)
    .pipe(a_categoria, ["estado"])
    .pipe(anotar, "zf_membresias", "3 estado recalculado"))

accesos_limpio = (zf_accesos
    .pipe(anotar, "zf_accesos", "0 original")
    .pipe(convertir_fechas, ["fecha_hora"], formato="%Y-%m-%d %H:%M:%S")
    .pipe(completar_dispositivo)
    .pipe(anotar, "zf_accesos", "1 tipos y dispositivo")
    .pipe(quitar_lecturas_dobles)
    .pipe(anotar, "zf_accesos", "2 sin lecturas dobles")
    .pipe(quitar_miembros_inexistentes, miembros_limpio)
    .pipe(quitar_fuera_de_horario)
    .pipe(anotar, "zf_accesos", "3 sin inexistentes ni fuera de horario")
    .pipe(asignar_membresia, membresias_limpio)
    .pipe(quitar_acceso_con_cancelada, membresias_limpio)
    .pipe(ajustar_enteros, {"id_acceso": "int32", "id_miembro": "int16"})
    .pipe(a_categoria, ["resultado", "dispositivo"])
    .pipe(anotar, "zf_accesos", "4 membresía asignada, sin canceladas"))

churn_limpio = (kg_churn
    .pipe(anotar, "kg_churn", "0 original")
    .pipe(limpiar_kg_churn)
    .pipe(anotar, "kg_churn", "1 nombres y tipos"))

actividad_limpio = (kg_actividad
    .pipe(anotar, "kg_actividad", "0 original")
    .pipe(limpiar_kg_actividad)
    .pipe(a_categoria, ["gender", "membership_type", "workout_type"])
    .pipe(anotar, "kg_actividad", "1 fechas y columnas"))

escenarios_limpio = (gov_escenarios
    .pipe(anotar, "gov_escenarios", "0 original")
    .pipe(limpiar_gov_escenarios)
    .pipe(anotar, "gov_escenarios", "1 texto y coordenadas"))

informe_limpieza = pd.DataFrame(registro)
informe_limpieza

,tabla,paso,filas,nulos,duplicados,memoria_kb
0,zf_planes,0 original,6,5,0,0.6
1,zf_planes,1 tipos,6,5,0,0.4
2,zf_miembros,0 original,908,127,0,95.1
3,zf_miembros,1 sin duplicados,900,126,0,94.6
4,zf_miembros,2 tipos y fechas de relleno,900,135,0,72.0
5,zf_miembros,3 categorías y faltantes,900,63,0,35.8
6,zf_pagos,0 original,3572,108,12,276.1
7,zf_pagos,1 sin duplicados,3560,107,0,276.5
8,zf_pagos,2 tipos,3560,107,0,180.8
9,zf_pagos,3 categorías y faltantes,3560,0,0,76.7


**Interpretación del informe de cambios**

- **`zf_miembros`:** quedan 900 miembros (se quitaron las 8 copias). Los nulos bajan de 127 a 63: los 63 que quedan son fechas de nacimiento (54 originales + 9 de relleno), que se conservan como faltantes a propósito. Pasar el texto a `category` reduce la memoria de 95,1 a 35,8 KB.
- **`zf_pagos`:** quedan 3.560 pagos (12 duplicados menos) y ningún nulo: los 107 métodos de pago vacíos pasan a "Sin dato" (el otro vacío estaba en una fila duplicada).
- **`zf_membresias`:** conserva sus 3.425 filas; los 51 precios faltantes se completan y 93 estados pasan de `Activa` a `Vencida`.
- **`zf_accesos`:** pasa de 57.003 a 56.109 accesos. Se eliminan 894 registros inválidos (675 lecturas dobles, 25 de miembros inexistentes, 15 fuera de horario y 179 con membresía cancelada). Los 386 nulos nuevos son la columna `id_membresia` de los accesos `Denegado`, que no tienen membresía vigente por definición. La memoria baja de 4.283 a 1.151 KB (73 % menos).
- **Externos:** `kg_churn` y `kg_actividad` no pierden filas y ocupan un tercio de la memoria; los 44 nulos de `gov_escenarios` son las 22 parejas de coordenadas fuera de Medellín que se anularon.

## 2.8 Validación de la limpieza

La función `validar_limpieza` convierte las reglas de calidad en comprobaciones automáticas con `assert`. Si alguna falla, el notebook se detiene y el mensaje dice qué regla no se cumple.

In [20]:
def validar_limpieza():
    m, ms, p, a = miembros_limpio, membresias_limpio, pagos_limpio, accesos_limpio

    # Miembros
    assert m["documento_hash"].is_unique, "documento repetido en miembros"
    assert m["id_miembro"].is_unique, "id_miembro repetido"
    assert set(m["genero"].cat.categories) <= {"M", "F", "Sin dato"}, "género sin unificar"
    assert not (m["comuna"] == "laureles").any(), "comuna sin unificar"
    assert m[["genero", "comuna"]].notna().all().all(), "faltantes en género o comuna"
    assert (m["fecha_nacimiento"].dropna() > pd.Timestamp("1900-01-01")).all(), "fecha de relleno"

    # Membresías
    assert ms["id_membresia"].is_unique, "id_membresia repetido"
    assert not ((ms["estado"] == "Activa") & (ms["fecha_fin"] < FECHA_CORTE)).any(), "activa vencida"
    assert ms["precio_pagado"].notna().all(), "precio faltante"
    assert (ms["fecha_fin"] >= ms["fecha_inicio"]).all(), "fecha_fin anterior a fecha_inicio"

    # Pagos
    assert p["id_pago"].is_unique, "id_pago repetido"
    assert pd.api.types.is_integer_dtype(p["valor"]), "valor no numérico"
    assert p["fecha_pago"].notna().all(), "fecha de pago sin convertir"
    assert p["metodo_pago"].notna().all(), "método de pago faltante"

    # Accesos
    segundos = a.sort_values(["id_miembro", "fecha_hora"]).groupby("id_miembro")["fecha_hora"].diff().dt.total_seconds()
    assert not (segundos < 60).any(), "quedan lecturas dobles"
    assert a["id_miembro"].isin(m["id_miembro"]).all(), "acceso de miembro inexistente"
    assert a["fecha_hora"].dt.hour.between(5, 21).all(), "acceso fuera de horario"
    permitidos = a[a["resultado"] == "Permitido"]
    assert permitidos["id_membresia"].notna().all(), "acceso permitido sin membresía vigente"
    estado = permitidos["id_membresia"].map(ms.set_index("id_membresia")["estado"])
    assert not (estado == "Cancelada").any(), "acceso permitido con membresía cancelada"
    assert a["dispositivo"].notna().all(), "dispositivo faltante"

    # Externos
    assert "attendance_status" not in actividad_limpio.columns, "columna no confiable"
    coords = escenarios_limpio[["latitud", "longitud"]].dropna()
    assert coords["latitud"].between(6.1, 6.4).all() and coords["longitud"].between(-75.75, -75.45).all()
    assert not escenarios_limpio["direccion"].str.contains("  ").any(), "espacios dobles en direcciones"

    return "Validación superada: todas las reglas de calidad se cumplen"


print(validar_limpieza())

Validación superada: todas las reglas de calidad se cumplen


## 2.9 Estado inicial y final de los datos

Se repite `resumen_calidad` sobre los datos limpios y se compara con el estado inicial.

In [21]:
datos_limpios = {
    "zf_planes": planes_limpio,
    "zf_miembros": miembros_limpio,
    "zf_membresias": membresias_limpio,
    "zf_pagos": pagos_limpio,
    "zf_accesos": accesos_limpio,
    "kg_churn": churn_limpio,
    "kg_actividad": actividad_limpio,
    "gov_escenarios": escenarios_limpio,
}
estado_final = resumen_calidad(datos_limpios)

comparacion_calidad = estado_inicial.merge(estado_final, on="dataset", suffixes=("_antes", "_despues"))
comparacion_calidad[["dataset", "filas_antes", "filas_despues", "nulos_antes", "nulos_despues",
                     "filas_duplicadas_antes", "filas_duplicadas_despues",
                     "columnas_texto_antes", "columnas_texto_despues",
                     "memoria_kb_antes", "memoria_kb_despues"]]

,dataset,filas_antes,filas_despues,nulos_antes,nulos_despues,filas_duplicadas_antes,filas_duplicadas_despues,columnas_texto_antes,columnas_texto_despues,memoria_kb_antes,memoria_kb_despues
0,zf_planes,6,6,5,5,0,0,2,1,0.6,0.4
1,zf_miembros,908,900,127,63,0,0,6,1,95.1,36.3
2,zf_membresias,3425,3425,51,0,0,0,3,0,277.4,100.6
3,zf_pagos,3572,3560,108,0,12,0,4,0,276.1,76.8
4,zf_accesos,57003,56109,1140,386,0,0,3,0,4283.0,1151.0
5,kg_churn,4000,4000,0,0,0,0,0,0,437.6,140.8
6,kg_actividad,2600,2600,0,0,0,0,6,0,317.0,112.0
7,gov_escenarios,1071,1071,0,44,0,0,3,3,128.7,125.1


In [22]:
def contar_problemas(o, l):
    """Cuenta cada problema del diagnóstico en los datos originales (o) y en los limpios (l)."""
    fecha_hora_o = pd.to_datetime(o["zf_accesos"]["fecha_hora"])
    ordenado = o["zf_accesos"].assign(fh=fecha_hora_o).sort_values(["id_miembro", "fh"])
    dobles_o = (ordenado.groupby("id_miembro")["fh"].diff().dt.total_seconds() < 60).sum()
    ordenado_l = l["zf_accesos"].sort_values(["id_miembro", "fecha_hora"])
    dobles_l = (ordenado_l.groupby("id_miembro")["fecha_hora"].diff().dt.total_seconds() < 60).sum()
    fin_o = pd.to_datetime(o["zf_membresias"]["fecha_fin"])
    return [
        ("Miembros con documento repetido", o["zf_miembros"]["documento_hash"].duplicated().sum(),
         l["zf_miembros"]["documento_hash"].duplicated().sum()),
        ("Valores de género distintos", o["zf_miembros"]["genero"].nunique(), l["zf_miembros"]["genero"].nunique()),
        ("Comuna 'laureles'", (o["zf_miembros"]["comuna"] == "laureles").sum(),
         (l["zf_miembros"]["comuna"] == "laureles").sum()),
        ("Fechas de nacimiento 1900-01-01", (o["zf_miembros"]["fecha_nacimiento"] == "1900-01-01").sum(),
         (l["zf_miembros"]["fecha_nacimiento"] == pd.Timestamp("1900-01-01")).sum()),
        ("Membresías 'Activa' vencidas", ((o["zf_membresias"]["estado"] == "Activa") & (fin_o < FECHA_CORTE)).sum(),
         ((l["zf_membresias"]["estado"] == "Activa") & (l["zf_membresias"]["fecha_fin"] < FECHA_CORTE)).sum()),
        ("Membresías sin precio", o["zf_membresias"]["precio_pagado"].isna().sum(),
         l["zf_membresias"]["precio_pagado"].isna().sum()),
        ("Pagos duplicados", o["zf_pagos"].duplicated().sum(), l["zf_pagos"].duplicated().sum()),
        ("Pagos con valor como texto", o["zf_pagos"]["valor"].str.contains("$", regex=False).sum(), 0),
        ("Pagos sin método", o["zf_pagos"]["metodo_pago"].isna().sum(), l["zf_pagos"]["metodo_pago"].isna().sum()),
        ("Accesos sin dispositivo", o["zf_accesos"]["dispositivo"].isna().sum(),
         l["zf_accesos"]["dispositivo"].isna().sum()),
        ("Lecturas dobles de huella", dobles_o, dobles_l),
        ("Accesos de miembros inexistentes",
         (~o["zf_accesos"]["id_miembro"].isin(o["zf_miembros"]["id_miembro"])).sum(),
         (~l["zf_accesos"]["id_miembro"].isin(l["zf_miembros"]["id_miembro"])).sum()),
        ("Accesos fuera de horario", ((fecha_hora_o.dt.hour < 5) | (fecha_hora_o.dt.hour >= 22)).sum(),
         (~l["zf_accesos"]["fecha_hora"].dt.hour.between(5, 21)).sum()),
        ("Escenarios con coordenadas fuera de Medellín",
         (~(o["gov_escenarios"]["latitud"].between(6.1, 6.4) & o["gov_escenarios"]["longitud"].between(-75.75, -75.45))).sum(),
         (~(l["gov_escenarios"]["latitud"].between(6.1, 6.4) & l["gov_escenarios"]["longitud"].between(-75.75, -75.45))
          & l["gov_escenarios"]["latitud"].notna()).sum()),
        ("Direcciones con espacios múltiples", o["gov_escenarios"]["direccion"].str.contains(r"\s{2,}").sum(),
         l["gov_escenarios"]["direccion"].str.contains(r"\s{2,}").sum()),
    ]


problemas = pd.DataFrame(contar_problemas(datos_originales, datos_limpios),
                         columns=["problema del diagnóstico", "antes", "después"])
problemas

,problema del diagnóstico,antes,después
0,Miembros con documento repetido,8,0
1,Valores de género distintos,6,3
2,Comuna 'laureles',18,0
3,Fechas de nacimiento 1900-01-01,9,0
4,Membresías 'Activa' vencidas,93,0
5,Membresías sin precio,51,0
6,Pagos duplicados,12,0
7,Pagos con valor como texto,179,0
8,Pagos sin método,108,0
9,Accesos sin dispositivo,1140,0


**Resultado de la Lección 2**

- Los 15 problemas medibles del diagnóstico quedaron en 0 (el género pasa de 6 a 3 valores: `M`, `F` y `Sin dato`).
- Los nulos que permanecen son decisiones documentadas: `ingresos_incluidos` de los planes ilimitados (no aplica), fechas de nacimiento desconocidas (no se inventan edades), membresía de los accesos denegados (no tenían) y coordenadas imposibles de escenarios.
- Las columnas de texto de Zona Fit pasan de 18 a 2 (`nombre_plan` y `documento_hash`, que son identificadores y no categorías): las fechas son fechas, los montos son números y las categorías son `category`.
- La memoria total de las tablas de Zona Fit baja de 4.932 KB a 1.365 KB (72 % menos).
- La validación con `assert` confirma que las reglas de calidad se cumplen, así que los datos están listos para el análisis.

# Lección 3. Filtrado, ordenamiento y agrupación de datos

En esta lección se usan los datos limpios para responder la pregunta analítica. Primero se traduce cada factor de la pregunta a la forma **"métrica por grupo"**:

| Factor de la pregunta | Métrica por grupo |
|---|---|
| Tipo de plan | Tasa de no renovación **por plan** |
| Frecuencia de asistencia | Tasa de no renovación **por visitas semanales** |
| Horario de entrenamiento | Tasa de no renovación **por franja horaria predominante** |
| Antigüedad / primeros seis meses | Tasa de no renovación **por antigüedad** y porcentaje de miembros que desertan en sus primeros 180 días |

## 3.1 Variable de respuesta: ¿la membresía se renovó?

Zona Fit no registra un "motivo de retiro", así que la deserción se mide con las fechas de las membresías:

- Una membresía **se renueva** si el mismo miembro compra otra que empieza **máximo 30 días** después de la fecha de fin.
- Una membresía **no se renueva** si fue **cancelada** o si pasaron más de 30 días desde su fin sin una nueva compra.
- Si la membresía todavía está vigente o venció hace menos de 30 días antes del corte, aún no se sabe qué pasará: **no es observable** y se excluye de las tasas.

Para calcularlo se ordenan las membresías de cada miembro por fecha y se toma, con `groupby().shift()`, la fecha de inicio de la siguiente.

In [23]:
def variable_respuesta(membresias, fecha_corte=FECHA_CORTE, dias_gracia=30):
    """Agrega a cada membresía si se renovó, si es observable y su antigüedad."""
    df = membresias.sort_values(["id_miembro", "fecha_inicio"]).reset_index(drop=True)
    df["siguiente_inicio"] = df.groupby("id_miembro")["fecha_inicio"].shift(-1)
    df["dias_hasta_siguiente"] = (df["siguiente_inicio"] - df["fecha_fin"]).dt.days
    cancelada = df["estado"] == "Cancelada"
    df["renovada"] = (df["dias_hasta_siguiente"] <= dias_gracia) & ~cancelada
    plazo_cumplido = df["fecha_fin"] + pd.Timedelta(days=dias_gracia) <= fecha_corte
    df["observable"] = df["renovada"] | plazo_cumplido | cancelada
    df["no_renovo"] = df["observable"] & ~df["renovada"]
    df["numero_membresia"] = df.groupby("id_miembro").cumcount() + 1
    alta = df.groupby("id_miembro")["fecha_inicio"].transform("min")
    df["antiguedad_dias"] = (df["fecha_inicio"] - alta).dt.days
    return df


analisis = variable_respuesta(membresias_limpio)
print("Membresías:", len(analisis))
print("Observables:", analisis["observable"].sum(), "| no observables:", (~analisis["observable"]).sum())
print(analisis.loc[analisis["observable"], "no_renovo"].value_counts())
print("Días entre membresías (solo renovaciones):")
print(analisis.loc[analisis["renovada"], "dias_hasta_siguiente"].describe().round(1))

Membresías: 3425
Observables: 3084 | no observables: 341
no_renovo
False    2428
True      656
Name: count, dtype: int64
Días entre membresías (solo renovaciones):
count    2428.0
mean        2.5
std         3.0
min         1.0
25%         1.0
50%         1.0
75%         2.0
max        12.0
Name: dias_hasta_siguiente, dtype: float64


Ahora se agregan a cada membresía los factores de la pregunta, usando los datos limpios:

- **Tipo de plan:** con `map` desde `planes_limpio`.
- **Frecuencia:** accesos permitidos de la membresía divididos entre las semanas que estuvo vigente (hasta la fecha de corte).
- **Horario:** franja en la que el miembro entró más veces durante la membresía. Las franjas se crean con `pd.cut`: mañana (5:00–11:59), tarde (12:00–17:59) y noche (18:00–21:59).

In [24]:
FRANJAS = {"bins": [4, 11, 17, 21], "labels": ["mañana", "tarde", "noche"]}

permitidos = accesos_limpio[accesos_limpio["resultado"] == "Permitido"].copy()
permitidos["franja"] = pd.cut(permitidos["fecha_hora"].dt.hour, **FRANJAS)

plan = planes_limpio.set_index("id_plan")
analisis["nombre_plan"] = analisis["id_plan"].map(plan["nombre_plan"])
analisis["tipo_plan"] = analisis["id_plan"].map(plan["tipo_plan"])

visitas = permitidos.groupby("id_membresia").size()
analisis["visitas"] = analisis["id_membresia"].map(visitas).fillna(0).astype("int16")
dias_vigente = (analisis["fecha_fin"].clip(upper=FECHA_CORTE) - analisis["fecha_inicio"]).dt.days + 1
analisis["visitas_semana"] = (analisis["visitas"] / (dias_vigente / 7)).round(2)

franja_principal = (permitidos.groupby(["id_membresia", "franja"], observed=True).size()
                              .sort_values(ascending=False)
                              .reset_index()
                              .drop_duplicates("id_membresia")
                              .set_index("id_membresia")["franja"])
analisis["franja_principal"] = analisis["id_membresia"].map(franja_principal)

analisis[["id_membresia", "id_miembro", "nombre_plan", "fecha_inicio", "fecha_fin", "visitas",
          "visitas_semana", "franja_principal", "numero_membresia", "renovada", "no_renovo"]].head()

,id_membresia,id_miembro,nombre_plan,fecha_inicio,fecha_fin,visitas,visitas_semana,franja_principal,numero_membresia,renovada,no_renovo
0,1,1,Mensual estudiante,2025-08-24,2025-09-22,5,1.17,mañana,1,False,True
1,2,1,Mensual estudiante,2026-02-14,2026-03-15,8,1.87,noche,2,True,False
2,3,1,Mensual estudiante,2026-03-16,2026-04-14,8,1.87,noche,3,True,False
3,4,1,Mensual estudiante,2026-04-24,2026-05-23,5,1.17,noche,4,True,False
4,5,1,Mensual estudiante,2026-05-24,2026-06-22,11,2.57,mañana,5,True,False


## 3.2 Filtrado con condiciones booleanas, `isin`, `between` y `query`

In [25]:
# Máscara booleana: solo las membresías cuyo resultado ya se conoce
observables = analisis[analisis["observable"]]
print("Membresías observables:", len(observables))

# isin: los dos planes mensuales
mensuales = observables[observables["nombre_plan"].isin(["Mensual ilimitado", "Mensual estudiante"])]
print("Membresías mensuales:", len(mensuales), f"({len(mensuales) / len(observables):.1%} de las observables)")

# between: accesos en la hora pico de la tarde (17:00 a 19:59)
hora_pico = permitidos[permitidos["fecha_hora"].dt.hour.between(17, 19)]
print("Accesos entre 17:00 y 19:59:", len(hora_pico), f"({len(hora_pico) / len(permitidos):.1%} de los accesos permitidos)")

# query: tiqueteras que no se renovaron y se usaron menos de una vez por semana
tiqueteras_riesgo = observables.query("tipo_plan == 'Tiquetera' and no_renovo and visitas_semana < 1")
print("Tiqueteras no renovadas con menos de 1 visita por semana:", len(tiqueteras_riesgo))

Membresías observables: 3084
Membresías mensuales: 2380 (77.2% de las observables)
Accesos entre 17:00 y 19:59: 17685 (31.7% de los accesos permitidos)
Tiqueteras no renovadas con menos de 1 visita por semana: 50


- El **77,2 %** de las membresías observables son planes mensuales: son el producto principal del gimnasio, y cada mes el miembro vuelve a decidir si sigue.
- Casi una de cada tres entradas (**31,7 %**) ocurre entre las 17:00 y las 19:59, la hora pico de la tarde.
- `query` permite escribir condiciones compuestas como texto; aquí aísla un grupo de riesgo: tiqueteras que no se renovaron y se usaron menos de una vez por semana.

## 3.3 Ordenamiento y selección de registros relevantes

In [26]:
visitas_miembro = permitidos.groupby("id_miembro").size()
print("Los 5 miembros con más visitas en todo el periodo:")
print(visitas_miembro.nlargest(5))

print("\nLas 5 membresías no renovadas con menos visitas por semana:")
display(observables[observables["no_renovo"]]
        .sort_values(["visitas_semana", "fecha_inicio"])
        [["id_miembro", "nombre_plan", "fecha_inicio", "fecha_fin", "visitas", "visitas_semana"]]
        .head())

print("Membresías más largas sin renovación (más días hasta volver):")
display(observables.nlargest(3, "dias_hasta_siguiente")
        [["id_miembro", "nombre_plan", "fecha_fin", "siguiente_inicio", "dias_hasta_siguiente"]])

Los 5 miembros con más visitas en todo el periodo:
id_miembro
209    370
672    350
383    331
706    328
834    328
dtype: int64

Las 5 membresías no renovadas con menos visitas por semana:


,id_miembro,nombre_plan,fecha_inicio,fecha_fin,visitas,visitas_semana
3336,870,Tiquetera 12 ingresos,2025-05-31,2025-06-29,0,0.0
1470,401,Tiquetera 12 ingresos,2025-12-06,2026-01-04,0,0.0
332,103,Anual ilimitado,2026-02-20,2027-02-19,0,0.0
1370,373,Mensual ilimitado,2026-03-11,2026-04-09,0,0.0
2191,580,Anual ilimitado,2026-03-11,2027-03-10,0,0.0


Membresías más largas sin renovación (más días hasta volver):


,id_miembro,nombre_plan,fecha_fin,siguiente_inicio,dias_hasta_siguiente
1440,392,Trimestral ilimitado,2026-02-02,2026-07-31,179.0
404,121,Mensual estudiante,2026-02-06,2026-08-01,176.0
834,236,Mensual ilimitado,2025-12-03,2026-05-27,175.0


`nlargest` y `sort_values` muestran los extremos: los miembros más constantes superan las 320 visitas en el periodo y, en el otro lado, hay membresías que no se renovaron sin una sola visita (las anuales de la lista son las canceladas: sus accesos se excluyeron en la limpieza). También hay miembros que vuelven después de casi seis meses: cuentan como no renovación de la membresía anterior y como una nueva compra.

## 3.4 Indicador 1: tasa de no renovación por tipo de plan

In [27]:
indicador_plan = (observables
    .groupby("nombre_plan")
    .agg(membresias=("id_membresia", "count"),
         no_renovaron=("no_renovo", "sum"),
         tasa_no_renovacion=("no_renovo", "mean"),
         precio_medio=("precio_pagado", "mean"))
    .sort_values("tasa_no_renovacion", ascending=False)
    .round({"tasa_no_renovacion": 3, "precio_medio": 0}))
indicador_plan

,membresias,no_renovaron,tasa_no_renovacion,precio_medio
nombre_plan,,,,
Anual ilimitado,26,10,0.385,836769.0
Tiquetera 12 ingresos,349,99,0.284,59622.0
Mensual estudiante,1092,233,0.213,69551.0
Mensual ilimitado,1288,260,0.202,89322.0
Trimestral ilimitado,253,46,0.182,238577.0
Semestral ilimitado,76,8,0.105,447039.0


In [28]:
indicador_tipo = (observables.groupby("tipo_plan", observed=True)
                             .agg(membresias=("id_membresia", "count"),
                                  tasa_no_renovacion=("no_renovo", "mean"))
                             .round(3))
indicador_tipo

,membresias,tasa_no_renovacion
tipo_plan,,
Ilimitado,2735,0.204
Tiquetera,349,0.284


**Interpretación**

- De las 3.084 membresías observables, **656 no se renovaron (21,3 %)**.
- La **tiquetera** es el plan con más deserción entre los de volumen alto: **28,4 %**, frente a **20,4 %** de los planes ilimitados. Quien compra por ingresos sueltos tiene menos compromiso con una rutina.
- Entre los ilimitados, la tasa baja a medida que el plan es más largo: mensual (20–21 %), trimestral (18,2 %) y semestral (10,5 %).
- El plan anual muestra 38,5 %, pero con solo 26 membresías observables (2 de ellas canceladas), así que no permite sacar conclusiones.

## 3.5 Indicador 2: frecuencia de asistencia

In [29]:
observables.groupby("no_renovo")["visitas_semana"].agg(["count", "mean", "median", "min", "max"]).round(2)

,count,mean,median,min,max
no_renovo,,,,,
False,2428,2.98,3.03,0.23,5.60
True,656,1.17,1.17,0.00,3.89


In [30]:
CATEGORIAS_FRECUENCIA = {"bins": [-0.01, 1, 2, 3, np.inf],
                         "labels": ["hasta 1", "más de 1 a 2", "más de 2 a 3", "más de 3"]}

observables = observables.assign(
    frecuencia=pd.cut(observables["visitas_semana"], **CATEGORIAS_FRECUENCIA))

indicador_frecuencia = (observables
    .groupby("frecuencia", observed=True)
    .agg(membresias=("id_membresia", "count"),
         tasa_no_renovacion=("no_renovo", "mean"))
    .round(3))
indicador_frecuencia

,membresias,tasa_no_renovacion
frecuencia,,
hasta 1,366,0.844
más de 1 a 2,578,0.483
más de 2 a 3,909,0.067
más de 3,1231,0.006


**Interpretación:** la frecuencia de asistencia es el factor con la diferencia más grande.

- Las membresías que no se renovaron tienen una mediana de **1,17 visitas por semana**, frente a **3,03** de las que sí se renovaron.
- Con **hasta 1 visita semanal**, el **84,4 %** no renueva; entre 1 y 2 visitas, el **48,3 %**; con más de 2, la tasa cae a **6,7 %**, y con más de 3 casi desaparece (**0,6 %**).
- El umbral práctico está en **dos visitas por semana**: por debajo de ese nivel, el miembro está en riesgo.

## 3.6 Indicador 3: horario de entrenamiento

In [31]:
indicador_franja = (observables
    .groupby("franja_principal", observed=True)
    .agg(membresias=("id_membresia", "count"),
         tasa_no_renovacion=("no_renovo", "mean"),
         visitas_semana_media=("visitas_semana", "mean"))
    .round(3))
indicador_franja

,membresias,tasa_no_renovacion,visitas_semana_media
franja_principal,,,
mañana,1480,0.211,2.612
tarde,946,0.196,2.656
noche,649,0.231,2.506


In [32]:
tabla_plan_franja = observables.pivot_table(index="nombre_plan", columns="franja_principal",
                                            values="no_renovo", aggfunc="mean", observed=True).round(3)
tabla_plan_franja

franja_principal,mañana,tarde,noche
nombre_plan,,,
Anual ilimitado,0.250,0.429,1.000
Mensual estudiante,0.221,0.206,0.204
Mensual ilimitado,0.188,0.174,0.262
Semestral ilimitado,0.089,0.100,0.100
Tiquetera 12 ingresos,0.286,0.318,0.188
Trimestral ilimitado,0.215,0.109,0.250


In [33]:
print("Distribución de los accesos permitidos por franja:")
print(permitidos["franja"].value_counts(normalize=True).round(3))

Distribución de los accesos permitidos por franja:
franja
mañana    0.377
tarde     0.334
noche     0.289
Name: proportion, dtype: float64


**Interpretación:** el horario influye poco.

- La tasa de no renovación varía solo entre **19,6 % (tarde)** y **23,1 % (noche)**. Quienes entrenan de noche van un poco menos (2,51 visitas por semana frente a 2,66 en la tarde), lo que explica parte de la diferencia.
- En la tabla dinámica se ve la diferencia más marcada en el plan mensual ilimitado: 26,2 % de no renovación de noche frente a 17,4 % en la tarde. Los valores extremos de los planes anuales y semestrales salen de grupos muy pequeños y no son concluyentes.
- La mañana concentra el 37,7 % de las entradas, la tarde el 33,4 % y la noche el 28,9 %.

## 3.7 Indicador 4: antigüedad y deserción en los primeros seis meses

### a) No renovación según la antigüedad del miembro

In [34]:
TRAMOS_ANTIGUEDAD = {"bins": [-1, 179, 364, np.inf],
                     "labels": ["0 a 6 meses", "6 a 12 meses", "más de 12 meses"]}

observables = observables.assign(
    tramo_antiguedad=pd.cut(observables["antiguedad_dias"], **TRAMOS_ANTIGUEDAD))

indicador_antiguedad = (observables
    .groupby("tramo_antiguedad", observed=True)
    .agg(membresias=("id_membresia", "count"),
         tasa_no_renovacion=("no_renovo", "mean"))
    .round(3))
indicador_antiguedad

,membresias,tasa_no_renovacion
tramo_antiguedad,,
0 a 6 meses,2268,0.239
6 a 12 meses,644,0.144
más de 12 meses,172,0.128


### b) Miembros que desertan en sus primeros seis meses

Este es el indicador central de la pregunta. Para cada miembro se toma la fecha de alta (su primera membresía) y la primera vez que no renovó. Es **desertor temprano** si eso ocurrió dentro de los primeros 180 días.

Solo se pueden evaluar los miembros que se dieron de alta al menos 210 días antes del corte (180 días + 30 de gracia); los demás todavía no completan su periodo. En las membresías canceladas no se registra la fecha de cancelación, así que se usa su fecha de inicio.

In [35]:
def desercion_temprana(analisis, fecha_corte=FECHA_CORTE, dias=180, dias_gracia=30):
    """Una fila por miembro con su alta, su primera no renovación y si desertó en los primeros `dias`."""
    eventos = analisis[analisis["no_renovo"]].assign(
        fecha_evento=lambda d: d["fecha_fin"].where(d["estado"] != "Cancelada", d["fecha_inicio"]))
    df = pd.DataFrame({
        "fecha_alta": analisis.groupby("id_miembro")["fecha_inicio"].min(),
        "primera_no_renovacion": eventos.groupby("id_miembro")["fecha_evento"].min(),
    })
    df["evaluable"] = df["fecha_alta"] <= fecha_corte - pd.Timedelta(days=dias + dias_gracia)
    df["desertor_6m"] = df["primera_no_renovacion"] <= df["fecha_alta"] + pd.Timedelta(days=dias)
    return df


desercion = desercion_temprana(analisis)
evaluables = desercion[desercion["evaluable"]]
print("Miembros:", len(desercion), "| evaluables:", len(evaluables))
print(f"Desertores en los primeros 6 meses: {evaluables['desertor_6m'].sum()} ({evaluables['desertor_6m'].mean():.1%})")

Miembros: 900 | evaluables: 671
Desertores en los primeros 6 meses: 410 (61.1%)


In [36]:
primera = analisis[analisis["numero_membresia"] == 1].set_index("id_miembro")
evaluables = evaluables.assign(
    plan_inicial=primera["nombre_plan"],
    frecuencia_inicial=pd.cut(primera["visitas_semana"], **CATEGORIAS_FRECUENCIA),
    franja_inicial=primera["franja_principal"])

for factor in ["plan_inicial", "frecuencia_inicial", "franja_inicial"]:
    tabla = (evaluables.groupby(factor, observed=True)["desertor_6m"]
                       .agg(miembros="count", tasa_desercion_6m="mean").round(3))
    display(tabla)

,miembros,tasa_desercion_6m
plan_inicial,,
Anual ilimitado,27,0.000
Mensual estudiante,221,0.710
Mensual ilimitado,240,0.667
Semestral ilimitado,34,0.147
Tiquetera 12 ingresos,79,0.747
Trimestral ilimitado,70,0.414


,miembros,tasa_desercion_6m
frecuencia_inicial,,
hasta 1,128,0.984
más de 1 a 2,149,0.792
más de 2 a 3,192,0.536
más de 3,202,0.312


,miembros,tasa_desercion_6m
franja_inicial,,
mañana,321,0.592
tarde,209,0.603
noche,139,0.662


**Interpretación:** los primeros meses son el periodo crítico.

- Las membresías compradas en los **primeros 6 meses** del miembro tienen **23,9 %** de no renovación, casi el doble que las de miembros con más de un año (**12,8 %**).
- **El 61,1 % de los miembros evaluables (410 de 671) dejó de renovar al menos una vez en sus primeros seis meses.** Algunos vuelven después, pero el gimnasio pierde ingresos en ese tiempo.
- **Por plan inicial:** la tiquetera (74,7 %) y los mensuales (66,7–71,0 %) tienen la deserción temprana más alta. El trimestral baja a 41,4 % y el semestral a 14,7 %. El anual queda en 0 % porque su vigencia cubre los seis meses completos: el contrato largo retiene por diseño, no necesariamente por preferencia.
- **Por frecuencia en la primera membresía:** casi todos los que van hasta una vez por semana desertan (98,4 %), frente a 31,2 % de los que van más de tres veces.
- **Por franja inicial:** la noche vuelve a quedar un poco por encima (66,2 % frente a 59–60 %).

## 3.8 Resumen de indicadores e interpretación

In [37]:
resumen_indicadores = pd.DataFrame([
    {"indicador": "Tasa de no renovación general",
     "valor": f"{observables['no_renovo'].mean():.1%}"},
    {"indicador": "Tasa de no renovación – Tiquetera vs. Ilimitado",
     "valor": f"{indicador_tipo.loc['Tiquetera', 'tasa_no_renovacion']:.1%} vs. "
              f"{indicador_tipo.loc['Ilimitado', 'tasa_no_renovacion']:.1%}"},
    {"indicador": "Visitas por semana (mediana) – no renovaron vs. renovaron",
     "valor": f"{observables.loc[observables['no_renovo'], 'visitas_semana'].median():.2f} vs. "
              f"{observables.loc[~observables['no_renovo'], 'visitas_semana'].median():.2f}"},
    {"indicador": "Tasa de no renovación – hasta 1 visita/semana vs. más de 3",
     "valor": f"{indicador_frecuencia['tasa_no_renovacion'].iloc[0]:.1%} vs. "
              f"{indicador_frecuencia['tasa_no_renovacion'].iloc[-1]:.1%}"},
    {"indicador": "Tasa de no renovación por franja (mañana / tarde / noche)",
     "valor": " / ".join(f"{t:.1%}" for t in indicador_franja["tasa_no_renovacion"])},
    {"indicador": "Tasa de no renovación – 0 a 6 meses vs. más de 12 meses de antigüedad",
     "valor": f"{indicador_antiguedad['tasa_no_renovacion'].iloc[0]:.1%} vs. "
              f"{indicador_antiguedad['tasa_no_renovacion'].iloc[-1]:.1%}"},
    {"indicador": "Miembros que desertan en sus primeros 6 meses",
     "valor": f"{evaluables['desertor_6m'].mean():.1%} de {len(evaluables)} evaluables"},
])
resumen_indicadores

,indicador,valor
0,Tasa de no renovación general,21.3%
1,Tasa de no renovación – Tiquetera vs. Ilimitado,28.4% vs. 20.4%
2,Visitas por semana (mediana) – no renovaron vs...,1.17 vs. 3.03
3,Tasa de no renovación – hasta 1 visita/semana ...,84.4% vs. 0.6%
4,Tasa de no renovación por franja (mañana / tar...,21.1% / 19.6% / 23.1%
5,Tasa de no renovación – 0 a 6 meses vs. más de...,23.9% vs. 12.8%
6,Miembros que desertan en sus primeros 6 meses,61.1% de 671 evaluables


**Relación con la pregunta analítica**

Los indicadores permiten ordenar los cuatro factores de la pregunta según su relación con la deserción:

1. **Frecuencia de asistencia (la más fuerte):** con menos de dos visitas por semana la no renovación pasa de menos del 7 % a casi la mitad, y con una visita o menos supera el 84 %.
2. **Antigüedad:** el riesgo se concentra en los primeros seis meses (23,9 % de no renovación frente a 12,8 % después del año), y el 61,1 % de los miembros evaluables deja de renovar en ese periodo.
3. **Tipo de plan:** la tiquetera y los planes mensuales tienen más deserción; los planes de 3 y 6 meses, menos.
4. **Horario de entrenamiento (la más débil):** quienes entrenan de noche desertan un poco más, en parte porque van con menos frecuencia.

Para el gimnasio, esto sugiere vigilar desde el primer mes a los miembros con menos de dos visitas semanales y ofrecer el paso de tiquetera o mensual a planes trimestrales o semestrales.

**Límites:** los datos de Zona Fit son simulados y los indicadores muestran asociaciones, no causas. Las conclusiones deben confirmarse con datos reales del gimnasio.